# 03 — GFP & Component Statistics

Component peak latencies and analysis
windows are loaded from `outputs/component_peaks.json` (written by notebook 02)
instead of being re-detected here — ensuring both notebooks use identical definitions.

Run notebook 02 first (Setup cells 0–4) to generate or refresh the JSON.


## 0. Imports


In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
import pingouin as pg

mne.set_log_level('WARNING')


## 1. Settings

Edit component search windows here. Each entry: `(name, polarity, tmin_ms, tmax_ms)`  
polarity `+1` = positive peak (argmax), `-1` = negative peak (argmin) on GFP.

In [ ]:
BIDS_ROOT = '/workspace/data/marmoset_FreqIntensity'

SUBJECTS = ['Cj399', 'Cj459']

# Reference label → (pipeline_name, gfp_exclude, amp_exclude)
# gfp_exclude : channels excluded from GFP (zero-valued ref channels only)
# amp_exclude : channels excluded from peak-channel reporting (same channels + any others)
REFERENCES = {
    'linkedEars': ('erp-pipeline-ref-linkedEars', [],       ['A1', 'A2']),
    'CAR':        ('erp-pipeline-ref-CAR',        [],       []),
    'CMR':        ('erp-pipeline-ref-CMR',        [],       []),
}

# Intensity and frequency levels to look for
INTENSITIES = ['45dB', '60dB', '75dB']
FREQUENCIES = ['00125Hz', '00250Hz', '00500Hz', '01000Hz',
               '02000Hz', '04000Hz', '08000Hz', '16000Hz']

LATENCY_INTENSITY = '75dB'   # intensity level used for peak-latency determination

# ── Component search windows (edit freely) ──────────────────────────────────
# (component_name, polarity, tmin_ms, tmax_ms)
COMPONENTS = [
    ('P1',  +1,  10,  20),
    ('N1',  -1,  30,  60),
    ('P2',  +1,  60,  100),
    ('N2',  -1, 120,  180),
    ('LN',  -1, 200,  600),   # late negativity
]

# ── Analysis window half-widths (ms) centred at peak latency ─────────────────
# Full window = peak_latency ± half-width
COMPONENT_WINDOW_MS = {
    'P1':  5 / 2,   #  5 ms wide
    'N1': 10 / 2,   # 10 ms wide
    'P2': 20 / 2,   # 20 ms wide
    'N2': 40 / 2,   # 40 ms wide
    'LN': 80 / 2,   # 80 ms wide
}
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = REPO_ROOT / 'outputs' / 'stats'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Helper functions

In [ ]:
def get_all_eeg_picks(evoked: mne.Evoked, gfp_exclude: list = []) -> np.ndarray:
    """EEG picks for GFP — excludes zero-valued ref channels (empty for linkedEars, CAR, CMR)."""
    return mne.pick_types(evoked.info, eeg=True, exclude=gfp_exclude)


def get_signal_picks(evoked: mne.Evoked, amp_exclude: list) -> np.ndarray:
    """EEG channel indices excluding reference electrodes — used for peak_channel reporting."""
    return mne.pick_types(evoked.info, eeg=True, exclude=amp_exclude)


def compute_gfp(data_uv: np.ndarray) -> np.ndarray:
    """GFP (population std, ddof=0) across channels at each time point.
    data_uv shape: (n_ch, n_times).
    """
    return data_uv.std(axis=0, ddof=0)


def find_peak_on_gfp(
    evoked: mne.Evoked,
    gfp: np.ndarray,
    data_uv: np.ndarray,
    picks: np.ndarray,
    tmin_ms: float,
    tmax_ms: float,
) -> dict:
    """
    Find the GFP maximum within [tmin_ms, tmax_ms].
    Returns latency, GFP amplitude, and the signal channel with the largest
    absolute amplitude at that latency.

    data_uv / picks: signal channels only (reference excluded).
    """
    times_ms = evoked.times * 1000
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)

    if not mask.any():
        return dict(latency_ms=np.nan, gfp_amplitude_uv=np.nan,
                    channel='', channel_amplitude_uv=np.nan)

    peak_idx = int(np.where(mask)[0][np.argmax(gfp[mask])])
    lat_ms   = float(times_ms[peak_idx])
    gfp_amp  = float(gfp[peak_idx])

    col     = data_uv[:, peak_idx]
    ch_idx  = int(np.argmax(np.abs(col)))
    ch_name = evoked.ch_names[picks[ch_idx]]
    ch_amp  = float(col[ch_idx])

    return dict(
        latency_ms=lat_ms,
        gfp_amplitude_uv=gfp_amp,
        channel=ch_name,
        channel_amplitude_uv=ch_amp,
    )


def load_evokeds_for_subject(
    bids_root: str,
    subject: str,
    pipeline: str,
    intensities: list,
    frequencies: list,
) -> dict:
    """Load all available condition evoked files for one subject.
    Returns {condition_label: mne.Evoked}, silently skipping missing files.
    """
    deriv = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    evokeds = {}
    for intensity in intensities:
        task = f'Passive{intensity}'
        for freq in frequencies:
            cond = f'{intensity}{freq}'
            safe_cond = cond.replace('_', '').replace('-', '')
            fname = deriv / f'sub-{subject}_task-{task}_cond-{safe_cond}_ave.fif'
            if fname.exists():
                try:
                    ev = mne.read_evokeds(fname, verbose=False)[0]
                    evokeds[cond] = ev
                except Exception as e:
                    print(f'  Warning: could not read {fname.name}: {e}')
    return evokeds

## 3. Load component peaks from JSON

Peak latencies and analysis windows are read from `outputs/component_peaks.json`
written by notebook 02, and expanded across all subjects and reference types
defined in `REFERENCES` above.


In [ ]:
_json_path = Path('/workspace/outputs/component_peaks.json')
if not _json_path.exists():
    raise FileNotFoundError(
        f'{_json_path} not found.\n'
        'Run notebook 04 (Setup cells 0-4) to generate it.'
    )

with open(_json_path) as _f:
    _comp_data = json.load(_f)

# Warn if any SUBJECTS are missing from the JSON
_missing_subjs = set(SUBJECTS) - set(_comp_data['subjects'])
if _missing_subjs:
    print(f'WARNING: subjects not in JSON: {_missing_subjs}')
    print(f'  Re-run notebook 04 with all subjects to update the JSON.')

# Reconstruct peak_latencies and analysis_windows for all subjects × references
# The same latencies (detected from linkedEars 75 dB) are applied to all refs.
peak_latencies  = {}
analysis_windows = {}

for _subj in SUBJECTS:
    if _subj not in _comp_data['subjects']:
        continue
    for ref_label in REFERENCES:
        for comp_name, _vals in _comp_data['subjects'][_subj].items():
            lat = _vals['latency_ms']
            w0  = _vals['window_tmin_ms']
            w1  = _vals['window_tmax_ms']
            lat = float('nan') if lat is None else float(lat)
            w0  = float('nan') if w0  is None else float(w0)
            w1  = float('nan') if w1  is None else float(w1)
            peak_latencies[  (_subj, ref_label, comp_name)] = lat
            analysis_windows[(_subj, ref_label, comp_name)] = (w0, w1)

print(f'Loaded: {_json_path}')
print(f'  Source: ref={_comp_data["latency_ref"]}, '
      f'intensity={_comp_data["latency_intensity"]}')
print()
print(f'{"Subject":<10} {"Ref":<12} {"Comp":<5}  {"Latency":>9}  {"Win tmin":>9}  {"Win tmax":>9}')
print('-' * 75)
for (_subj, _ref, comp_name), (_w0, _w1) in analysis_windows.items():
    _lat = peak_latencies[(_subj, _ref, comp_name)]
    _lat_s = f'{_lat:8.1f}' if not np.isnan(_lat) else '     n/a'
    _w0_s  = f'{_w0:9.1f}' if not np.isnan(_w0)  else '     n/a'
    _w1_s  = f'{_w1:9.1f}' if not np.isnan(_w1)  else '     n/a'
    print(f'{_subj:<10} {_ref:<12} {comp_name:<5}  {_lat_s}  {_w0_s}  {_w1_s}')


## 4. Step 2 — Measure amplitudes at fixed latencies across all conditions

In [ ]:
all_rows = []

for ref_label, (pipeline, gfp_exclude, amp_exclude) in REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        # Only load 75 dB — latencies were determined from this condition
        evokeds = load_evokeds_for_subject(
            BIDS_ROOT, subject, pipeline, [LATENCY_INTENSITY], FREQUENCIES
        )
        if not evokeds:
            print(f'  sub-{subject}: no evoked files — skipping')
            continue

        print(f'  sub-{subject}: {len(evokeds)} conditions')

        for cond_label, evoked in evokeds.items():
            try:
                db_end    = cond_label.index('dB') + 2
                intensity = cond_label[:db_end]
                frequency = cond_label[db_end:]

                all_picks = get_all_eeg_picks(evoked, gfp_exclude)
                gfp       = compute_gfp(evoked.data[all_picks] * 1e6)

                sig_picks = get_signal_picks(evoked, amp_exclude)
                sig_data  = evoked.data[sig_picks] * 1e6
                times_ms  = evoked.times * 1000

                for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
                    lat_ms = peak_latencies.get((subject, ref_label, comp_name), np.nan)

                    if np.isnan(lat_ms):
                        gfp_amp = ch_name = ch_amp = np.nan
                    else:
                        t_idx   = int(np.argmin(np.abs(times_ms - lat_ms)))
                        gfp_amp = float(gfp[t_idx])
                        col     = sig_data[:, t_idx]
                        ch_idx  = int(np.argmax(np.abs(col)))
                        ch_name = evoked.ch_names[sig_picks[ch_idx]]
                        ch_amp  = float(col[ch_idx])

                    all_rows.append(dict(
                        subject=subject,
                        reference=ref_label,
                        condition=cond_label,
                        intensity=intensity,
                        frequency=frequency,
                        component=comp_name,
                        search_tmin_ms=tmin_ms,
                        search_tmax_ms=tmax_ms,
                        latency_ms=lat_ms,
                        gfp_amplitude_uv=gfp_amp,
                        peak_channel=ch_name,
                        peak_channel_amplitude_uv=ch_amp,
                        nave=evoked.nave,
                    ))
            except Exception as e:
                print(f'  Warning: sub-{subject} {cond_label} failed: {e}')
                continue

df = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df)}')
df.head(10)

## 5. Save to TSV

In [ ]:
# Per-subject per-reference files
for ref_label in df['reference'].unique():
    for subject in df['subject'].unique():
        subset = df[(df['reference'] == ref_label) & (df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_peak-latencies.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

# Combined file
combined_path = OUTPUT_DIR / 'all_subjects_peak-latencies.tsv'
df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(df)} rows)')

## 6. GFP plot with component markers

Visual sanity check — mean 75 dB GFP with detected peak latencies and search windows.  
Edit `PLOT_SUBJECT`, `PLOT_REF`, and `PLOT_FREQUENCIES` to inspect different subjects/references/conditions.  
Set `PLOT_FREQUENCIES = None` to average across all 8 frequencies.

In [ ]:
PLOT_SUBJECT     = 'Cj399'
PLOT_REF         = 'linkedEars'
# PLOT_FREQUENCIES = ['01000Hz']  # None = all 8 frequencies; e.g. ['01000Hz'] for one condition
PLOT_FREQUENCIES = None  # None = all 8 frequencies; e.g. ['01000Hz'] for one condition

pipeline, gfp_exclude, amp_exclude = REFERENCES[PLOT_REF]

plot_freqs = PLOT_FREQUENCIES if PLOT_FREQUENCIES is not None else FREQUENCIES
evokeds_75 = load_evokeds_for_subject(
    BIDS_ROOT, PLOT_SUBJECT, pipeline, [LATENCY_INTENSITY], plot_freqs
)
if not evokeds_75:
    print(f'No 75 dB evokeds for sub-{PLOT_SUBJECT}, ref-{PLOT_REF}')
else:
    gfp_stack = []
    ref_evoked = None
    for ev in evokeds_75.values():
        all_picks = get_all_eeg_picks(ev, gfp_exclude)
        data_uv   = ev.data[all_picks] * 1e6
        gfp_stack.append(compute_gfp(data_uv))
        ref_evoked = ev
    mean_gfp = np.mean(gfp_stack, axis=0)
    times_ms = ref_evoked.times * 1000

    freq_label = ', '.join(plot_freqs) if len(plot_freqs) <= 3 else f'{len(plot_freqs)} freq'
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.plot(times_ms, mean_gfp, color='k', lw=1.5,
            label=f'GFP ({freq_label}, {LATENCY_INTENSITY})')
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlim(-100, 650)
    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('GFP (µV)')
    ax.set_title(f'sub-{PLOT_SUBJECT}  ref-{PLOT_REF}  {LATENCY_INTENSITY}  {freq_label}  '
                 f'(all EEG ch, ddof=0)')

    comp_colors = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
                   'N2': '#FF9800', 'LN': '#9C27B0'}

    for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
        col = comp_colors.get(comp_name, 'gray')
        lat = peak_latencies.get((PLOT_SUBJECT, PLOT_REF, comp_name), np.nan)
        if np.isnan(lat):
            continue

        # Analysis window (light fill)
        w0, w1 = analysis_windows.get((PLOT_SUBJECT, PLOT_REF, comp_name), (np.nan, np.nan))
        if not np.isnan(w0):
            ax.axvspan(w0, w1, alpha=0.15, color=col, zorder=1)

        # Peak latency marker + label
        amp = float(mean_gfp[np.argmin(np.abs(times_ms - lat))])
        ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
        ax.text(lat + 2, amp + 0.05, comp_name, color=col, fontsize=8,
                va='bottom', zorder=3)

    fig.tight_layout()
    plt.show()

    SAVE_DIR = '/workspace/figures'
    _default_name = f'sub-{PLOT_SUBJECT}_ref-{PLOT_REF}_GFP'
    import ipywidgets as _w
    from IPython.display import display as _display
    _name = _w.Text(value=_default_name,
                    description='Filename:', layout=_w.Layout(width='360px'))
    _fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                        description='Format:', layout=_w.Layout(width='160px'))
    _btn  = _w.Button(description='Save', button_style='primary',
                      layout=_w.Layout(width='80px'))
    _out  = _w.Output()
    _fig_ref = fig

    def _save(_):
        with _out:
            _out.clear_output()
            path = f'{SAVE_DIR}/{_name.value}.{_fmt.value}'
            _fig_ref.savefig(path, format=_fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn.on_click(_save)
    _display(_w.HBox([_name, _fmt, _btn]), _out)


## 6b. GFP overlay by intensity level

One line per intensity (GFP averaged across all 8 frequency conditions).  
Component windows and peak latencies are taken from Section 3.  
Edit `PLOT7B_SUBJECT` and `PLOT7B_REF` to switch subject or reference.

In [ ]:
# ── Section 7b settings ───────────────────────────────────────────────────────
PLOT7B_SUBJECT = 'Cj399'
PLOT7B_REF     = 'linkedEars'   # must be a key in REFERENCES
# ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = '/workspace/figures'

_int_colors_7b  = {'45dB': '#2196F3', '60dB': '#FF9800', '75dB': '#F44336'}
_comp_colors_7b = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
                   'N2': '#FF9800', 'LN': '#9C27B0'}

pipeline, gfp_exclude, amp_exclude = REFERENCES[PLOT7B_REF]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlim(-100, 650)

ref_ev_7b = None
for intensity in INTENSITIES:
    evokeds = load_evokeds_for_subject(
        BIDS_ROOT, PLOT7B_SUBJECT, pipeline, [intensity], FREQUENCIES
    )
    if not evokeds:
        print(f'No evokeds for sub-{PLOT7B_SUBJECT} {intensity} — skipping')
        continue

    # Average GFP across all 8 frequency conditions (mirrors Section 3 approach)
    gfp_stack = []
    for ev in evokeds.values():
        picks = get_all_eeg_picks(ev, gfp_exclude)
        gfp_stack.append(compute_gfp(ev.data[picks] * 1e6))
        ref_ev_7b = ev
    mean_gfp_7b = np.mean(gfp_stack, axis=0)
    ax.plot(ref_ev_7b.times * 1000, mean_gfp_7b,
            color=_int_colors_7b[intensity], lw=1.8, label=intensity)

if ref_ev_7b is not None:
    times_ms_7b = ref_ev_7b.times * 1000
    for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
        col = _comp_colors_7b.get(comp_name, 'gray')
        lat = peak_latencies.get((PLOT7B_SUBJECT, PLOT7B_REF, comp_name), np.nan)
        if np.isnan(lat):
            continue
        w0, w1 = analysis_windows.get(
            (PLOT7B_SUBJECT, PLOT7B_REF, comp_name), (np.nan, np.nan)
        )
        if not np.isnan(w0):
            ax.axvspan(w0, w1, alpha=0.12, color=col, zorder=1)
        ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
        ax.text(lat + 2, 0.97, comp_name, color=col, fontsize=8, va='top',
                transform=ax.get_xaxis_transform(), zorder=3)

ax.set_xlabel('Time (ms)')
ax.set_ylabel('GFP (µV)')
ax.set_title(
    f'sub-{PLOT7B_SUBJECT}  ref-{PLOT7B_REF}  — GFP by intensity (avg over {len(FREQUENCIES)} frequencies)'
)
ax.legend(title='Intensity', fontsize=9, title_fontsize=9)
fig.tight_layout()
plt.show()


_default_name = f'sub-{PLOT7B_SUBJECT}_ref-{PLOT7B_REF}_GFP-by-intensity'
import ipywidgets as _w
from IPython.display import display as _display
_name = _w.Text(value=_default_name,
                description='Filename:', layout=_w.Layout(width='360px'))
_fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                    description='Format:', layout=_w.Layout(width='160px'))
_btn  = _w.Button(description='Save', button_style='primary',
                  layout=_w.Layout(width='80px'))
_out  = _w.Output()
_fig_ref = fig

def _save(_):
    with _out:
        _out.clear_output()
        path = f'{SAVE_DIR}/{_name.value}.{_fmt.value}'
        _fig_ref.savefig(path, format=_fmt.value, bbox_inches='tight')
        print(f'Saved: {path}')

_btn.on_click(_save)
_display(_w.HBox([_name, _fmt, _btn]), _out)


## 6c. GFP overlay by frequency — one figure per intensity

One figure per intensity level; each figure overlays GFP waveforms for all 8 frequency conditions.
Component windows and peak latencies are taken from Section 3.
Edit `PLOT6C_SUBJECT` and `PLOT6C_REF` to switch subject or reference.

In [ ]:
# ── Section 6c settings ───────────────────────────────────────────────────────
PLOT6C_SUBJECT = 'Cj399'
PLOT6C_REF     = 'linkedEars'   # must be a key in REFERENCES
# ─────────────────────────────────────────────────────────────────────────────
SAVE_DIR = '/workspace/figures'

_freq_colors_6c = {
    f: c for f, c in zip(FREQUENCIES, [
        '#e6194b','#3cb44b','#4363d8','#f58231',
        '#911eb4','#42d4f4','#f032e6','#bfef45',
    ])
}
_comp_colors_6c = {'P1': '#2196F3', 'N1': '#F44336', 'P2': '#4CAF50',
                   'N2': '#FF9800', 'LN': '#9C27B0'}

pipeline6c, gfp_exclude_6c, amp_exclude_6c = REFERENCES[PLOT6C_REF]

# Compute y-axis limits from the 75 dB condition so all figures share the same scale
_ev75_all = load_evokeds_for_subject(
    BIDS_ROOT, PLOT6C_SUBJECT, pipeline6c, ['75dB'], FREQUENCIES
)
if _ev75_all:
    _gfp75_vals = []
    for _ev in _ev75_all.values():
        _picks75 = get_all_eeg_picks(_ev, gfp_exclude_6c)
        _gfp75_vals.append(compute_gfp(_ev.data[_picks75] * 1e6))
    _gfp75_all = np.concatenate(_gfp75_vals)
    _pad75 = (_gfp75_all.max() - _gfp75_all.min()) * 0.05
    _ylim6c = (_gfp75_all.min() - _pad75, _gfp75_all.max() + _pad75)
else:
    _ylim6c = None

import ipywidgets as _w
from IPython.display import display as _display

for intensity in INTENSITIES:
    evokeds = load_evokeds_for_subject(
        BIDS_ROOT, PLOT6C_SUBJECT, pipeline6c, [intensity], FREQUENCIES
    )
    if not evokeds:
        print(f'No evokeds for sub-{PLOT6C_SUBJECT} {intensity} — skipping')
        continue

    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.axvline(0, color='gray', lw=0.8, linestyle='--')
    ax.set_xlim(-100, 650)
    if _ylim6c is not None:
        ax.set_ylim(_ylim6c)

    ref_ev_6c = None
    for freq in FREQUENCIES:
        cond_label = f'{intensity}{freq}'
        ev = evokeds.get(cond_label)
        if ev is None:
            continue
        picks = get_all_eeg_picks(ev, gfp_exclude_6c)
        gfp   = compute_gfp(ev.data[picks] * 1e6)
        hz    = int(freq.replace('Hz', ''))
        lbl   = f'{hz // 1000}k Hz' if hz >= 1000 else f'{hz} Hz'
        ax.plot(ev.times * 1000, gfp,
                color=_freq_colors_6c[freq], lw=1.4, label=lbl)
        ref_ev_6c = ev

    if ref_ev_6c is not None:
        times_ms_6c = ref_ev_6c.times * 1000
        for comp_name, polarity, tmin_ms, tmax_ms in COMPONENTS:
            col = _comp_colors_6c.get(comp_name, 'gray')
            lat = peak_latencies.get((PLOT6C_SUBJECT, PLOT6C_REF, comp_name), np.nan)
            if np.isnan(lat):
                continue
            w0, w1 = analysis_windows.get(
                (PLOT6C_SUBJECT, PLOT6C_REF, comp_name), (np.nan, np.nan)
            )
            if not np.isnan(w0):
                ax.axvspan(w0, w1, alpha=0.12, color=col, zorder=1)
            ax.axvline(lat, color=col, lw=1.2, linestyle=':', zorder=2)
            ax.text(lat + 2, 0.97, comp_name, color=col, fontsize=8,
                    va='top', transform=ax.get_xaxis_transform(), zorder=3)

    ax.set_xlabel('Time (ms)')
    ax.set_ylabel('GFP (µV)')
    ax.set_title(
        f'sub-{PLOT6C_SUBJECT}  ref-{PLOT6C_REF}  {intensity}'
        f' — GFP by frequency'
    )
    ax.legend(title='Frequency', fontsize=8, title_fontsize=8,
              loc='upper right', ncol=2)
    fig.tight_layout()
    plt.show()

    _default_name = f'sub-{PLOT6C_SUBJECT}_ref-{PLOT6C_REF}_GFP-by-freq_{intensity}'
    _name = _w.Text(value=_default_name,
                    description='Filename:', layout=_w.Layout(width='360px'))
    _fmt  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                        description='Format:', layout=_w.Layout(width='160px'))
    _btn  = _w.Button(description='Save', button_style='primary',
                      layout=_w.Layout(width='80px'))
    _out  = _w.Output()

    def _save(_, _fig=fig, _out=_out, _name=_name, _fmt=_fmt):
        with _out:
            _out.clear_output()
            path = f'{SAVE_DIR}/{_name.value}.{_fmt.value}'
            _fig.savefig(path, format=_fmt.value, bbox_inches='tight')
            print(f'Saved: {path}')

    _btn.on_click(_save)
    _display(_w.HBox([_name, _fmt, _btn]), _out)


---

## 7. Two-way ANOVA (Intensity × Frequency): GFP

For each **reference × subject × component**:

1. Load clean epoch files (one per intensity level).
2. For each trial, compute mean GFP over the component's analysis window.
3. Run a two-way ANOVA with factors **Intensity** (45 dB, 75 dB) and **Frequency** (8 levels).

Each trial is one independent observation. Intensity and Frequency are between-trial factors
(each trial belongs to exactly one Intensity × Frequency cell).
Unequal trial counts across cells are handled by Type III SS (pingouin default) — no trial-count balancing needed.

Output: `outputs/stats/sub-<ID>_ref-<ref>_anova.tsv` + combined `all_subjects_anova.tsv`

Columns: `reference`, `subject`, `component`, `source`, `ddof1`, `ddof2`, `F`, `p`, `np2`

In [ ]:
def load_epochs_for_subject(bids_root, subject, pipeline, intensities):
    """Load clean epoch files; return {intensity: mne.Epochs}. Missing files skipped."""
    deriv = Path(bids_root) / 'derivatives' / pipeline / f'sub-{subject}' / 'eeg'
    epochs_by_intensity = {}
    for intensity in intensities:
        fif = deriv / f'sub-{subject}_task-Passive{intensity}_desc-clean_epo.fif'
        if fif.exists():
            epo = mne.read_epochs(fif, preload=True, verbose=False)
            epochs_by_intensity[intensity] = epo
        else:
            print(f'  Missing epoch file: {fif.name}')
    return epochs_by_intensity


def mean_gfp_over_window(epochs, all_picks, tmin_ms, tmax_ms):
    """
    Return mean GFP amplitude (µV) per trial over [tmin_ms, tmax_ms].
    epochs.get_data() shape: (n_trials, n_ch, n_times)
    """
    times_ms = epochs.times * 1000
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)
    data = epochs.get_data(picks=all_picks) * 1e6   # (n_trials, n_ch, n_times)
    # GFP per trial per time: std across channels (ddof=0)
    gfp = data.std(axis=1, ddof=0)                  # (n_trials, n_times)
    return gfp[:, mask].mean(axis=1)                # (n_trials,)

In [ ]:
anova_rows = []

for ref_label, (pipeline, gfp_exclude, amp_exclude) in REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels loaded — skipping ANOVA')
            continue

        ref_epo   = next(iter(epochs_by_intensity.values()))
        all_picks = get_all_eeg_picks(ref_epo)

        for comp_name, polarity, search_tmin, search_tmax in COMPONENTS:
            w0, w1 = analysis_windows.get((subject, ref_label, comp_name), (np.nan, np.nan))
            if np.isnan(w0):
                print(f'    {comp_name}: no analysis window — skipping')
                continue

            # Build long-form DataFrame: one row per trial
            rows = []
            for intensity, epo in epochs_by_intensity.items():
                for freq in FREQUENCIES:
                    cond_label = f'{intensity}{freq}'
                    if cond_label not in epo.event_id:
                        continue
                    cond_epo = epo[cond_label]
                    if len(cond_epo) == 0:
                        continue
                    amps = mean_gfp_over_window(cond_epo, all_picks, w0, w1)
                    for amp in amps:
                        rows.append(dict(intensity=intensity, frequency=freq, amplitude=amp))

            if not rows:
                print(f'    {comp_name}: no data — skipping')
                continue

            long_df = pd.DataFrame(rows)

            try:
                aov = pg.anova(
                    data=long_df,
                    dv='amplitude',
                    between=['intensity', 'frequency'],
                    detailed=True,
                )
                # ddof2 is the residual DF, shared across all effect rows
                ddof2 = int(aov.loc[aov['Source'] == 'Residual', 'DF'].iloc[0])

                effects = aov[aov['Source'] != 'Residual']
                for _, aov_row in effects.iterrows():
                    anova_rows.append(dict(
                        reference=ref_label,
                        subject=subject,
                        component=comp_name,
                        source=aov_row['Source'],
                        ddof1=int(aov_row['DF']),
                        ddof2=ddof2,
                        F=round(float(aov_row['F']), 4),
                        p=round(float(aov_row['p_unc']), 6),
                        np2=round(float(aov_row['np2']), 4),
                    ))

                summary = '  '.join(
                    f"{r['Source']} F={r['F']:.2f} p={r['p_unc']:.4f}"
                    for _, r in effects.iterrows()
                )
                print(f'    {comp_name}: {summary}')
            except Exception as e:
                print(f'    {comp_name}: ANOVA failed — {e}')

anova_df = pd.DataFrame(anova_rows)
print(f'\nTotal ANOVA result rows: {len(anova_df)}')
anova_df.head(15)

### 7b. Save ANOVA results to TSV

In [ ]:
# Per-subject per-reference files
for ref_label in anova_df['reference'].unique():
    for subject in anova_df['subject'].unique():
        subset = anova_df[(anova_df['reference'] == ref_label) & (anova_df['subject'] == subject)]
        if subset.empty:
            continue
        fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_anova_GFP.tsv'
        subset.to_csv(fname, sep='\t', index=False)
        print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')

# Combined file
combined_path = OUTPUT_DIR / 'all_subjects_anova_GFP.tsv'
anova_df.to_csv(combined_path, sep='\t', index=False)
print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(anova_df)} rows)')

---

## 8. Simple-effects ANOVAs (intensity × frequency)

For each **reference × subject × component**, two sets of one-way ANOVAs are run on the
same GFP data as Section 7:

- **Intensity simple effect at each frequency** — one-way ANOVA (factor = intensity, 3 levels)
  separately for each of the 8 frequency conditions.
- **Frequency simple effect at each intensity** — one-way ANOVA (factor = frequency, 8 levels)
  separately for each of the 3 intensity conditions.

`*` in the printed summary marks p < 0.05.  
The full long-form data is stored in `sec10_balanced` and reused by **Section 10b** (next cell)
for pairwise frequency comparisons.

Output: `outputs/stats/sub-<ID>_ref-<ref>_simple_effects_GFP.tsv`

In [ ]:
# ── Section 9 settings ──────────────────────────────────────────────────────
SEC10_AMPLITUDE = 'GFP'   # 'GFP' only; per-channel not implemented here
SEC10_PADJUST   = 'bonf'  # p-adjustment used in Section 10b pairwise tests
# ─────────────────────────────────────────────────────────────────────────────

_freq_lbl = {}
for _f in FREQUENCIES:
    _hz = int(_f.replace('Hz', ''))
    _freq_lbl[_f] = f'{_hz // 1000}k' if _hz >= 1000 else str(_hz)

sec10_rows     = []   # simple-effect ANOVA results
sec10_balanced = {}   # (ref_label, subject, comp_name) → balanced DataFrame

for ref_label, (pipeline, gfp_exclude, amp_exclude) in REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if len(epochs_by_intensity) < 2:
            print(f'  sub-{subject}: fewer than 2 intensity levels — skipping')
            continue

        ref_epo      = next(iter(epochs_by_intensity.values()))
        picks_for_dv = get_all_eeg_picks(ref_epo)

        for comp_name, polarity, search_tmin, search_tmax in COMPONENTS:
            w0, w1 = analysis_windows.get((subject, ref_label, comp_name), (np.nan, np.nan))
            if np.isnan(w0):
                continue

            rows = []
            for intensity, epo in epochs_by_intensity.items():
                for freq in FREQUENCIES:
                    cond_label = f'{intensity}{freq}'
                    if cond_label not in epo.event_id:
                        continue
                    cond_epo = epo[cond_label]
                    if len(cond_epo) == 0:
                        continue
                    amps = mean_gfp_over_window(cond_epo, picks_for_dv, w0, w1)
                    for amp in amps:
                        rows.append(dict(intensity=intensity, frequency=freq, amplitude=amp))

            if not rows:
                continue

            long_df = pd.DataFrame(rows)
            sec10_balanced[(ref_label, subject, comp_name)] = long_df

            print(f'    {comp_name}:')

            # ── Intensity simple effect at each frequency ─────────────────────
            freq_marks = []
            for freq in FREQUENCIES:
                subset = long_df[long_df['frequency'] == freq]
                try:
                    aov     = pg.anova(data=subset, dv='amplitude', between='intensity', detailed=True)
                    err_src = 'Within' if 'Within' in aov['Source'].values else 'Residual'
                    p_col_a = 'p_unc'
                    df1 = int(aov.loc[aov['Source'] == 'intensity', 'DF'].iloc[0])
                    df2 = int(aov.loc[aov['Source'] == err_src,     'DF'].iloc[0])
                    F   = float(aov.loc[aov['Source'] == 'intensity', 'F'].iloc[0])
                    p   = float(aov.loc[aov['Source'] == 'intensity', p_col_a].iloc[0])
                    np2 = float(aov.loc[aov['Source'] == 'intensity', 'np2'].iloc[0])
                    sec10_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        test_factor='intensity', fixed_factor='frequency', fixed_level=freq,
                        ddof1=df1, ddof2=df2, F=round(F, 4), p=round(p, 6), np2=round(np2, 4),
                    ))
                    freq_marks.append(_freq_lbl[freq] + ('*' if p < 0.05 else ''))
                except Exception as e:
                    print(f'      intensity@{freq}: failed — {e}')
                    freq_marks.append(_freq_lbl[freq] + '!')
            print(f'      intensity @ freq:     {" ".join(freq_marks)}')

            # ── Frequency simple effect at each intensity ─────────────────────
            int_marks = []
            for intensity in INTENSITIES:
                subset = long_df[long_df['intensity'] == intensity]
                try:
                    aov     = pg.anova(data=subset, dv='amplitude', between='frequency', detailed=True)
                    err_src = 'Within' if 'Within' in aov['Source'].values else 'Residual'
                    p_col_a = 'p_unc'
                    df1 = int(aov.loc[aov['Source'] == 'frequency', 'DF'].iloc[0])
                    df2 = int(aov.loc[aov['Source'] == err_src,     'DF'].iloc[0])
                    F   = float(aov.loc[aov['Source'] == 'frequency', 'F'].iloc[0])
                    p   = float(aov.loc[aov['Source'] == 'frequency', p_col_a].iloc[0])
                    np2 = float(aov.loc[aov['Source'] == 'frequency', 'np2'].iloc[0])
                    sec10_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        test_factor='frequency', fixed_factor='intensity', fixed_level=intensity,
                        ddof1=df1, ddof2=df2, F=round(F, 4), p=round(p, 6), np2=round(np2, 4),
                    ))
                    int_marks.append(intensity + ('*' if p < 0.05 else ''))
                except Exception as e:
                    print(f'      frequency@{intensity}: failed — {e}')
                    int_marks.append(intensity + '!')
            print(f'      frequency @ intensity: {" ".join(int_marks)}')

sec10_df = pd.DataFrame(sec10_rows)
print(f'\nSimple-effect ANOVA rows: {len(sec10_df)}')
if len(sec10_df):
    display(sec10_df.head(10))

# ── Save ─────────────────────────────────────────────────────────────────────
if not sec10_df.empty:
    for ref_label in sec10_df['reference'].unique():
        for subject in sec10_df['subject'].unique():
            subset = sec10_df[
                (sec10_df['reference'] == ref_label) & (sec10_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_simple_effects_GFP.tsv'
            subset.to_csv(fname, sep='\t', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / 'all_subjects_simple_effects_GFP.tsv'
    sec10_df.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(sec10_df)} rows)')


In [ ]:
## 8b. Pairwise frequency comparisons at each intensity level
# Reuses `sec10_balanced` built above — no epoch re-loading needed.
# For each intensity, tests all pairs of the 8 frequency conditions (28 pairs × 3 intensities).
# ─────────────────────────────────────────────────────────────────────────────

_p_adj_col = 'p_corr' if SEC10_PADJUST != 'none' else 'p_unc'

sec10b_rows = []

for ref_label, (pipeline, *_) in REFERENCES.items():
    print(f'\n=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'\n  sub-{subject}')
        for comp_name, *_ in COMPONENTS:
            key = (ref_label, subject, comp_name)
            if key not in sec10_balanced:
                continue
            balanced = sec10_balanced[key]

            print(f'    {comp_name}:')
            for intensity in INTENSITIES:
                subset = balanced[balanced['intensity'] == intensity].copy()
                try:
                    ph = pg.pairwise_tests(
                        data=subset, dv='amplitude', between='frequency',
                        padjust=SEC10_PADJUST,
                    )
                except Exception as e:
                    print(f'      {intensity}: pairwise failed — {e}')
                    continue

                p_unc_col    = 'p_unc' if 'p_unc' in ph.columns else 'p-unc'
                actual_p_col = 'p_corr' if 'p_corr' in ph.columns else p_unc_col

                for _, ph_row in ph.iterrows():
                    sec10b_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        intensity=intensity,
                        padjust=SEC10_PADJUST,
                        A=ph_row['A'], B=ph_row['B'],
                        T=round(float(ph_row['T']), 4),
                        dof=round(float(ph_row['dof']), 2),
                        p_unc=round(float(ph_row.get(p_unc_col, np.nan)), 6),
                        p_adj=round(float(ph_row.get('p_corr', ph_row.get(actual_p_col, np.nan))), 6),
                        hedges=round(float(ph_row.get('hedges', np.nan)), 4),
                    ))

                sig = ph[ph[actual_p_col] < 0.05]
                n_sig, n_pairs = len(sig), len(ph)
                print(f'      {intensity}: {n_sig}/{n_pairs} pairs p_adj<.05')

sec10b_df = pd.DataFrame(sec10b_rows)
print(f'\nPairwise frequency rows: {len(sec10b_df)}')
if len(sec10b_df):
    display(sec10b_df.head(10))

# ── 1000 Hz comparisons ───────────────────────────────────────────────────
if len(sec10b_df):
    mask_1k = (sec10b_df['A'] == '01000Hz') | (sec10b_df['B'] == '01000Hz')
    df_1k = sec10b_df[mask_1k].copy().reset_index(drop=True)
    out_1k = OUTPUT_DIR / 'all_subjects_pairwise_freq_GFP_1000Hz.tsv'
    df_1k.to_csv(out_1k, sep='\t', index=False)
    print(f'Saved 1000 Hz pairs ({len(df_1k)} rows): {out_1k.relative_to(REPO_ROOT)}')
    display(df_1k)

# ── Save ──────────────────────────────────────────────────────────────────────
for label, out_df, tag in [
    ('pairwise frequency',  sec10b_df, 'pairwise_freq_GFP'),
]:
    if out_df.empty:
        print(f'No {label} results to save.')
        continue
    for ref_label in out_df['reference'].unique():
        for subject in out_df['subject'].unique():
            subset = out_df[
                (out_df['reference'] == ref_label) & (out_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_{tag}.tsv'
            subset.to_csv(fname, sep='\t', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / f'all_subjects_{tag}.tsv'
    out_df.to_csv(combined_path, sep='\t', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(out_df)} rows)')


In [ ]:
## 8c. Pairwise intensity comparisons at each frequency level
# Reuses `sec10_balanced` built above — no epoch re-loading needed.
# For each frequency, tests all pairs of the 3 intensity conditions (3 pairs × 8 frequencies).
# ─────────────────────────────────────────────────────────────────────────────

sec10c_rows = []

for ref_label, (pipeline, *_) in REFERENCES.items():
    print(f'=== reference: {ref_label} ===')

    for subject in SUBJECTS:
        print(f'sub-{subject}')
        for comp_name, *_ in COMPONENTS:
            key = (ref_label, subject, comp_name)
            if key not in sec10_balanced:
                continue
            balanced = sec10_balanced[key]

            print(f'    {comp_name}:')
            for freq in FREQUENCIES:
                subset = balanced[balanced['frequency'] == freq].copy()
                try:
                    ph = pg.pairwise_tests(
                        data=subset, dv='amplitude', between='intensity',
                        padjust=SEC10_PADJUST,
                    )
                except Exception as e:
                    print(f'      {freq}: pairwise failed — {e}')
                    continue

                p_unc_col    = 'p_unc' if 'p_unc' in ph.columns else 'p-unc'
                actual_p_col = 'p_corr' if 'p_corr' in ph.columns else p_unc_col

                for _, ph_row in ph.iterrows():
                    sec10c_rows.append(dict(
                        reference=ref_label, subject=subject, component=comp_name,
                        frequency=freq,
                        padjust=SEC10_PADJUST,
                        A=ph_row['A'], B=ph_row['B'],
                        T=round(float(ph_row['T']), 4),
                        dof=round(float(ph_row['dof']), 2),
                        p_unc=round(float(ph_row.get(p_unc_col, float('nan'))), 6),
                        p_adj=round(float(ph_row.get('p_corr', ph_row.get(actual_p_col, float('nan')))), 6),
                        hedges=round(float(ph_row.get('hedges', float('nan'))), 4),
                    ))

                sig = ph[ph[actual_p_col] < 0.05]
                n_sig, n_pairs = len(sig), len(ph)
                print(f'      {freq}: {n_sig}/{n_pairs} pairs p_adj<.05')

sec10c_df = pd.DataFrame(sec10c_rows)
print(f'Pairwise intensity rows: {len(sec10c_df)}')
if len(sec10c_df):
    display(sec10c_df.head(10))

# ── Save ──────────────────────────────────────────────────────────────────────
for label, out_df, tag in [
    ('pairwise intensity', sec10c_df, 'pairwise_intensity_GFP'),
]:
    if out_df.empty:
        print(f'No {label} results to save.')
        continue
    for ref_label in out_df['reference'].unique():
        for subject in out_df['subject'].unique():
            subset = out_df[
                (out_df['reference'] == ref_label) & (out_df['subject'] == subject)
            ]
            if subset.empty:
                continue
            fname = OUTPUT_DIR / f'sub-{subject}_ref-{ref_label}_{tag}.tsv'
            subset.to_csv(fname, sep='	', index=False)
            print(f'Saved: {fname.relative_to(REPO_ROOT)}  ({len(subset)} rows)')
    combined_path = OUTPUT_DIR / f'all_subjects_{tag}.tsv'
    out_df.to_csv(combined_path, sep='	', index=False)
    print(f'Saved: {combined_path.relative_to(REPO_ROOT)}  (combined, {len(out_df)} rows)')

---

## 9. Amplitude vs frequency by intensity level

For each subject × reference, one figure with 5 subplots (one per component).  
X-axis: tone frequency (log scale, 125–16 000 Hz).  
Y-axis: GFP amplitude (or peak-channel ERP amplitude) at the fixed peak latency.  
Lines: one per intensity level (45 dB, 60 dB, 75 dB).

Peak latencies are taken from Section 3 (estimated from the 75 dB grand-average GFP).  
Set `PLOT11_AMPLITUDE` to `'GFP'` or `'peak_channel'`.

In [ ]:
# ── Section 9 settings ──────────────────────────────────────────────────────
PLOT11_AMPLITUDE = 'GFP'   # 'GFP' → GFP amplitude; 'peak_channel' → fixed peak-channel ERP
CI_LEVEL = 0.95             # confidence level for error bands
SAVE_DIR = '/workspace/figures'
# ─────────────────────────────────────────────────────────────────────────────

import scipy.stats as _stats
import ipywidgets as _w
from IPython.display import display as _display

def ci_half(values, level=0.95):
    """Half-width of the t-based CI for a 1-D array."""
    n = len(values)
    if n < 2:
        return np.nan
    se = values.std(ddof=1) / np.sqrt(n)
    t  = _stats.t.ppf((1 + level) / 2, df=n - 1)
    return t * se

freq_hz = [int(f.replace('Hz', '')) for f in FREQUENCIES]
intensity_colors = {'45dB': '#2196F3', '60dB': '#4EB458', '75dB': '#F44336'}
y_label = 'GFP amplitude (µV)' if PLOT11_AMPLITUDE == 'GFP' else 'ERP amplitude (µV)'

for ref_label, (pipeline, gfp_exclude, amp_exclude) in REFERENCES.items():
    for subject in SUBJECTS:

        epochs_by_intensity = load_epochs_for_subject(
            BIDS_ROOT, subject, pipeline, INTENSITIES
        )
        if not epochs_by_intensity:
            print(f'sub-{subject} ref-{ref_label}: no epoch files — skipping')
            continue

        ref_epo   = next(iter(epochs_by_intensity.values()))
        all_picks = get_all_eeg_picks(ref_epo)
        sig_picks = get_signal_picks(ref_epo, amp_exclude)

        # ── Peak channel per component (from evoked, 75 dB average) ──────────
        evokeds_75  = load_evokeds_for_subject(
            BIDS_ROOT, subject, pipeline, [LATENCY_INTENSITY], FREQUENCIES
        )
        times_ms    = ref_epo.times * 1000
        sig_mean_75 = np.mean(
            [ev.data[sig_picks] * 1e6 for ev in evokeds_75.values()], axis=0
        ) if evokeds_75 else None

        peak_ch = {}
        for comp_name, *_ in COMPONENTS:
            lat = peak_latencies.get((subject, ref_label, comp_name), np.nan)
            if np.isnan(lat) or sig_mean_75 is None:
                peak_ch[comp_name] = None
                continue
            t_idx  = int(np.argmin(np.abs(times_ms - lat)))
            ch_idx = int(np.argmax(np.abs(sig_mean_75[:, t_idx])))
            peak_ch[comp_name] = ref_epo.ch_names[sig_picks[ch_idx]]

        # ── Extract per-trial amplitude per cell ──────────────────────────────
        # amp_trials[(intensity, freq, comp)] = 1-D array of per-trial amplitudes
        amp_trials = {}
        for intensity, epo in epochs_by_intensity.items():
            for freq in FREQUENCIES:
                cond_label = f'{intensity}{freq}'
                if cond_label not in epo.event_id:
                    continue
                cond_epo = epo[cond_label]
                if len(cond_epo) == 0:
                    continue

                for comp_name, *_ in COMPONENTS:
                    w0, w1 = analysis_windows.get(
                        (subject, ref_label, comp_name), (np.nan, np.nan)
                    )
                    if np.isnan(w0):
                        continue

                    if PLOT11_AMPLITUDE == 'GFP':
                        trials = mean_gfp_over_window(cond_epo, all_picks, w0, w1)
                    else:
                        ch = peak_ch.get(comp_name)
                        if ch is None:
                            continue
                        ch_pick = [ref_epo.ch_names.index(ch)]
                        trials  = mean_erp_over_window(cond_epo, ch_pick, w0, w1).ravel()

                    amp_trials[(intensity, freq, comp_name)] = trials

        # ── Plot ──────────────────────────────────────────────────────────────
        n_comp = len(COMPONENTS)
        fig, axes = plt.subplots(1, n_comp, figsize=(3.2 * n_comp, 4), sharey=False)
        fig.suptitle(
            f'sub-{subject}  ref-{ref_label}  —  {y_label} ± {int(CI_LEVEL*100)}% CI\n'
            f'(mean over analysis window per trial)',
            fontsize=10, y=1.02,
        )

        for ax, (comp_name, polarity, *_) in zip(axes, COMPONENTS):
            for intensity in INTENSITIES:
                col = intensity_colors[intensity]
                means    = []
                half_cis = []
                for freq in FREQUENCIES:
                    trials = amp_trials.get((intensity, freq, comp_name), np.array([]))
                    means.append(np.mean(trials) if len(trials) > 0 else np.nan)
                    half_cis.append(ci_half(trials, CI_LEVEL) if len(trials) > 1 else np.nan)

                means    = np.array(means)
                half_cis = np.array(half_cis)

                ax.fill_between(freq_hz, means - half_cis, means + half_cis,
                                color=col, alpha=0.18)
                ax.plot(freq_hz, means,
                        color=col, marker='o', markersize=4, lw=1.5, label=intensity)

            ax.set_xscale('log')
            ax.set_xticks(freq_hz)
            ax.set_xticklabels(['125', '250', '500', '1k', '2k', '4k', '8k', '16k'],
                               rotation=45, ha='right', fontsize=7)

            ch = peak_ch.get(comp_name)
            if PLOT11_AMPLITUDE == 'peak_channel' and ch:
                ax.set_title(f'{comp_name} ({ch})', fontsize=10)
            else:
                ax.set_title(comp_name, fontsize=10)

            if PLOT11_AMPLITUDE == 'peak_channel' and polarity == -1:
                ax.invert_yaxis()

            ax.set_xlabel('Frequency (Hz)', fontsize=8)
            ax.grid(True, which='both', alpha=0.3, lw=0.5)

        axes[0].set_ylabel(y_label, fontsize=8)
        axes[-1].legend(title='Intensity', fontsize=8, title_fontsize=8)
        fig.tight_layout()
        plt.show()

        _fig_s  = fig
        _name_s = _w.Text(value=f'sub-{subject}_ref-{ref_label}_GFP-vs-freq',
                          description='Filename:', layout=_w.Layout(width='400px'))
        _fmt_s  = _w.Dropdown(options=['pdf', 'svg', 'png'], value='pdf',
                              description='Format:', layout=_w.Layout(width='160px'))
        _btn_s  = _w.Button(description='Save', button_style='primary',
                            layout=_w.Layout(width='80px'))
        _out_s  = _w.Output()

        def _save_s(_, f=_fig_s, n=_name_s, fmt=_fmt_s, o=_out_s):
            with o:
                o.clear_output()
                path = f'{SAVE_DIR}/{n.value}.{fmt.value}'
                f.savefig(path, format=fmt.value, bbox_inches='tight')
                print(f'Saved: {path}')

        _btn_s.on_click(_save_s)
        _display(_w.HBox([_name_s, _fmt_s, _btn_s]), _out_s)
